In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVC


In [9]:
# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv("Titanic-Dataset.csv")

# Display 5 random rows to see what the data looks like
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
290,291,1,1,"Barber, Miss. Ellen ""Nellie""",female,26.0,0,0,19877,78.8500,NaN,S
794,795,0,3,"Dantcheff, Mr. Ristiu",male,25.0,0,0,349203,7.8958,NaN,S
723,724,0,2,"Hodges, Mr. Henry Price",male,50.0,0,0,250643,13.0000,NaN,S
419,420,0,3,"Van Impe, Miss. Catharina",female,10.0,0,2,345773,24.1500,NaN,S
483,484,1,3,"Turkula, Mrs. (Hedwig)",female,63.0,0,0,4134,9.5875,NaN,S


In [10]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [11]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
637,638,0,2,"Collyer, Mr. Harvey",male,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
550,551,1,1,"Thayer, Mr. John Borland Jr",male,17.0,0,2,17421,110.8833,C70,C,3,C
149,150,0,2,"Byles, Rev. Thomas Roussel Davids",male,42.0,0,0,244310,13.0000,Missing,S,1,M
874,875,1,2,"Abelson, Mrs. Samuel (Hannah Wizosky)",female,28.0,1,0,P/PP 3381,24.0000,Missing,C,2,M
501,502,0,3,"Canavan, Miss. Mary",female,21.0,0,0,364846,7.7500,Missing,Q,1,M


In [12]:
df['Deck'].value_counts()

,count
Deck,
M,687
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


In [13]:
# Separate the features (X) from the target we want to predict (y)
# We drop 'Survived' from X because that's our answer key
X = df.drop('Survived', axis=1)

# y contains only the survival status
y = df['Survived']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,3,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,1,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,3,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,3,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,1,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [16]:
y_train

,Survived
692,1
481,0
527,0
855,1
801,1
...,...
359,1
258,1
736,0
462,0


In [17]:
#age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

/tmp/ipykernel_445/987313337.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


In [18]:
# pipeline

# numerical
p1 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

In [19]:
# categories goes one after another via stricly following the order of the input
OrdinalEncoder(
    categories=[[3, 2, 1]]
)

OrdinalEncoder(categories=[[3, 2, 1]])

In [20]:
# pipeline

#  categorical columns

p3 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),

        ('scaler',MinMaxScaler())
    ]
)


In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'
)
preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

In [22]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

#Model Training and Hyperparameter Tuning

We don't just pick a model; we optimize it. We use GridSearchCV to test different settings (kernels and C values) for our Support Vector Classifier. This automatically finds the best configuration using Cross-Validation.

In [23]:
SVC_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', SVC())
    ]
)

## 8. Final Evaluation

Finally, we evaluate the model using three key metrics:
*   **Accuracy**: Overall correctness.
*   **Precision**: Out of those predicted to survive, how many actually did?
*   **Recall**: Out of all actual survivors, how many did the model find?

In [24]:
from sklearn.model_selection import GridSearchCV


In [25]:
grid_param =[ {
    "model__kernel" : ['linear'],
    "model__C" :[0.01 , 0.1 , 1 , 10 , 50 , 100]
},
  {
      "model__kernel" : ['rbf'] ,
      "model__C" :[0.01 , 0.1 ,1,100],
      "model__gamma" :[0.01,0.1,5,10,'scale','auto']
  },
    {
        "model__kernel": ['poly'],
        "model__C" :[0.01 , 0.1 , 1 ,100],
        "model__degree" : [2,3]
    }

]


In [26]:
best_SVC_model = GridSearchCV(
    estimator=SVC_model,
    param_grid=grid_param,
    cv=5
)



In [27]:
best_SVC_model.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('pipeline_1',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age']),
                                                                        ('pipeline_2',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Fare',
                                                                          'Family_Size']),
                                                                        ('pipeline_3',
                                                                         Pipeline(steps=[('im...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Pclass'])])),
                                       ('model', SVC())]),
             param_grid=[{'model__C': [0.01, 0.1, 1, 10, 50, 100],
                          'model__kernel': ['linear']},
                         {'model__C': [0.01, 0.1, 1, 100],
                          'model__gamma': [0.01, 0.1, 5, 10, 'scale', 'auto'],
                          'model__kernel': ['rbf']},
                         {'model__C': [0.01, 0.1, 1, 100],
                          'model__degree': [2, 3], 'model__kernel': ['poly']}])

In [28]:
best_SVC_model

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('pipeline_1',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age']),
                                                                        ('pipeline_2',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Fare',
                                                                          'Family_Size']),
                                                                        ('pipeline_3',
                                                                         Pipeline(steps=[('im...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Pclass'])])),
                                       ('model', SVC())]),
             param_grid=[{'model__C': [0.01, 0.1, 1, 10, 50, 100],
                          'model__kernel': ['linear']},
                         {'model__C': [0.01, 0.1, 1, 100],
                          'model__gamma': [0.01, 0.1, 5, 10, 'scale', 'auto'],
                          'model__kernel': ['rbf']},
                         {'model__C': [0.01, 0.1, 1, 100],
                          'model__degree': [2, 3], 'model__kernel': ['poly']}])

In [29]:
SVC().get_params().keys() # valid perameter dekar opai


dict_keys(['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma', 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose'])

In [30]:

from sklearn.metrics import accuracy_score, precision_score, recall_score
y_pred_train = best_SVC_model.predict(X_train)

print(accuracy_score(y_train,y_pred_train))

0.8551483420593369


In [31]:
y_pred = best_SVC_model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)
print(accuracy)
precision = precision_score(y_test,y_pred)
print(precision)
recall = recall_score(y_test,y_pred)
print(recall)

0.7932960893854749
0.9
0.5217391304347826


কোন সমস্যা নেই। এবার আমি **একদম শুরু থেকে** বুঝাই। ধরে নাও তুমি SVM সম্পর্কে কিছুই জানো না।

---

## প্রথমে SVM কী করে?

ধরো তোমার কাছে দুই ধরনের ফল আছে।

🍎 আপেল = 0

🍊 কমলা = 1

ডেটা এমন:

```text
🍎 🍎 🍎 🍎

🍊 🍊 🍊 🍊
```

SVM-এর কাজ হলো **একটা Line (Boundary)** আঁকা, যাতে দুই গ্রুপ আলাদা হয়ে যায়।

```text
🍎 🍎 🍎 🍎
------------------
🍊 🍊 🍊 🍊
```

এই Line-কেই বলে **Decision Boundary**।

---

# এখন `kernel` কী?

Kernel ঠিক করে **Boundary কেমন হবে**।

### 1. Linear

```python
kernel="linear"
```

Boundary হবে **সোজা Line**।

```
🍎 🍎 🍎

------------

🍊 🍊 🍊
```

যখন Data সহজে আলাদা করা যায় তখন Linear ব্যবহার করা হয়।

---

### 2. RBF

ধরো Data এমন:

```
      🍎 🍎

   🍎      🍎

      🍊

   🍊      🍊
```

এখানে সোজা Line দিয়ে আলাদা করা যাবে?

❌ না।

তখন RBF ব্যবহার করা হয়।

RBF Curve এঁকে আলাদা করে।

```
      🍎 🍎
   🍎      🍎
    (      )
      🍊
   🍊      🍊
```

---

### 3. Polynomial

এটাও Curve ব্যবহার করে।

কিন্তু Curve-এর Shape Polynomial Formula দিয়ে তৈরি হয়।

---

# এবার `C` বুঝি

ধরো Teacher Exam নিচ্ছে।

একজন Student ভুল করেছে।

Teacher দুই রকম হতে পারে।

---

### ছোট C

```python
C=0.01
```

Teacher বলছে

> "এক-দুইটা ভুল করলে সমস্যা নেই।"

Model-ও তাই করবে।

কিছু ভুল Accept করবে।

Boundary হবে সহজ।

---

### বড় C

```python
C=100
```

Teacher বলছে

> "একটাও ভুল চলবে না।"

Model চেষ্টা করবে সব Training Data ঠিক করতে।

Boundary অনেক বাঁকা হতে পারে।

---

সহজ ভাষায়

| C ছোট         | C বড়            |
| ------------- | ---------------- |
| ভুল মেনে নেয় | ভুল মেনে নেয় না |

---

# এবার `gamma`

এটাই সবাই প্রথমে বুঝতে কষ্ট পায়।

ধরো একটা Street Light আছে।

### ছোট Gamma

```
      💡

🌕🌕🌕🌕🌕🌕🌕🌕
```

Light অনেক দূর পর্যন্ত যায়।

একটা Point অনেক দূরের Data-কেও প্রভাবিত করে।

Boundary Smooth হয়।

---

### বড় Gamma

```
    💡

   🌕
```

Light শুধু কাছাকাছি যায়।

Point শুধু নিজের আশেপাশে প্রভাব ফেলে।

Boundary অনেক বাঁকা হয়ে যায়।

---

এজন্য

```
Gamma ছোট
↓

Smooth Boundary
```

```
Gamma বড়
↓

Complex Boundary
```

---

# এবার `degree`

শুধু Polynomial-এর জন্য।

ধরো

Degree = 2

```
🙂
 \__
```

Degree = 3

```
🙂
 \__/\
```

Degree যত বাড়বে

Curve তত বেশি বাঁকবে।

---

# এবার `GridSearchCV` কী করে?

ধরো তুমি Mobile কিনবে।

তোমার তিনটা Choice আছে।

```
Samsung

iPhone

Xiaomi
```

তুমি সবগুলো Compare করবে।

যেটা ভালো

সেটা কিনবে।

GridSearchCV-ও একই কাজ করে।

এটা সব Parameter Try করে।

যেটা Best Accuracy দেয়

সেটা Select করে।

---

# এবার তোমার Code-এর দিকে আসি

```python
"model__kernel":["linear"]
```

মানে

> প্রথমে Linear Kernel দিয়ে Train করো।

---

```python
"model__C":[0.01,0.1,1,10,50,100]
```

মানে

```
প্রথমে C=0.01

তারপর C=0.1

তারপর C=1

তারপর C=10

তারপর C=50

তারপর C=100
```

প্রতিটা দিয়ে আলাদা Model বানাবে।

---

```python
"model__gamma":[0.01,0.1,5,10,"scale","auto"]
```

মানে

RBF Kernel-এর জন্য

```
gamma=0.01

gamma=0.1

gamma=5

gamma=10

gamma=scale

gamma=auto
```

সব Try করবে।

---

```python
cv=5
```

মানে

ডেটাকে ৫ ভাগ করবে।

```
Part1

Part2

Part3

Part4

Part5
```

একবার Part1 Test হবে।

একবার Part2 Test হবে।

...

এভাবে ৫ বার Train হবে।

তারপর Average Accuracy বের করবে।

---

## সবশেষে একটি সহজ উদাহরণ

ধরো GridSearchCV-এর কাছে এই Parameter আছে:

```python
kernel = ["linear", "rbf"]

C = [1, 10]
```

তাহলে এটি **৪টি Model** তৈরি করবে:

| Model | Kernel | C  |
| ----- | ------ | -- |
| 1     | Linear | 1  |
| 2     | Linear | 10 |
| 3     | RBF    | 1  |
| 4     | RBF    | 10 |

এগুলো সব Train করবে, Accuracy তুলনা করবে, তারপর যেটা সবচেয়ে ভালো হবে সেটাকেই **Best Model** হিসেবে বেছে নেবে।



হ্যাঁ, **`model__C`-এর `C` হলো SVC-এর fixed parameter name**, কিন্তু **`model` fixed না**।

চলো ভেঙে দেখি।

---

## `model__C`

এখানে দুইটা অংশ আছে:

```text
model__C
│      │
│      └── SVC-এর parameter
└───────── Pipeline-এর step-এর নাম
```

### `model`

এটা **তুমি যেটা নাম দিয়েছ**।

যেমন:

```python
SVC_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', SVC())
    ]
)
```

এখানে step-এর নাম **`model`**।

তাই লিখবে

```python
model__C
```

---

যদি তুমি এমন লিখতে:

```python
Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', SVC())
    ]
)
```

তাহলে লিখতে হবে

```python
classifier__C
classifier__kernel
classifier__gamma
```

---

## `C`

এটা **SVC-এর fixed parameter name**।

এটা পরিবর্তন করা যাবে না।

এগুলো সব fixed:

```python
C
kernel
gamma
degree
probability
shrinking
```

---

## উদাহরণ

```python
Pipeline(
    steps=[
        ('abc', SVC())
    ]
)
```

তাহলে হবে

```python
abc__C
abc__kernel
abc__gamma
```

---

### মনে রাখার সহজ নিয়ম

```text
step_name__parameter_name
```

অর্থাৎ

```text
model__C
│       │
│       └── SVC-এর fixed parameter
└────────── Pipeline-এ তোমার দেওয়া step-এর নাম
```

### তুমি কীভাবে সব valid parameter দেখবে?

```python
SVC().get_params().keys()
```

এটি চালালে SVC-এর সব fixed parameter-এর নাম দেখতে পাবে, যেমন `C`, `kernel`, `gamma`, `degree` ইত্যাদি।
